# NYC Airbnb - Exploratory Data Analysis (EDA)

**Dataset:** `cleaned_airbnb_final.csv` - 102,301 listings across 5 NYC boroughs  
**Objective:** Uncover pricing patterns, borough-level trends, room type distributions, and review behavior through statistical analysis and visualizations.

### Analysis Sections
1. Data Overview and Descriptive Statistics
2. Price Distribution Analysis
3. Borough-Level Comparison
4. Room Type Breakdown
5. Correlation Heatmap
6. Availability and Review Patterns
7. Top Neighbourhoods Deep Dive
8. Key Insights Summary

In [ ]:
# ---------------------------------------------------------------------------
# Import libraries and configure visualization settings
# ---------------------------------------------------------------------------
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# Set global plot style for consistent dark-themed visualizations
plt.style.use("dark_background")
sns.set_palette("Set2")
plt.rcParams.update({
    "figure.figsize": (12, 5),
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "figure.facecolor": "#0f1117",
    "axes.facecolor": "#0f1117",
    "savefig.facecolor": "#0f1117"
})

# Define a consistent color mapping for each borough
BOROUGH_COLORS = {
    "Manhattan": "#6366f1",
    "Brooklyn": "#06b6d4",
    "Queens": "#10b981",
    "Bronx": "#f59e0b",
    "Staten Island": "#f43f5e"
}

# Load the cleaned dataset
df = pd.read_csv("cleaned_airbnb_final.csv")
print(f"Loaded {len(df):,} rows x {df.shape[1]} columns")
df.head()

---
## 1 · Descriptive Statistics

In [ ]:
# Numeric summary
key_cols = ['price', 'service_fee', 'minimum_nights', 'number_of_reviews',
            'reviews_per_month', 'review_rate_number', 'availability_365']
df[key_cols].describe().round(2)

In [ ]:
# Categorical summary
print('=== Borough Distribution ===')
borough_pct = (df['neighbourhood_group'].value_counts(normalize=True) * 100).round(1)
for b, pct in borough_pct.items():
    print(f'  {b:15s} {pct:5.1f}%  ({df[df.neighbourhood_group==b].shape[0]:,} listings)')

print('\n=== Room Type Distribution ===')
room_pct = (df['room_type'].value_counts(normalize=True) * 100).round(1)
for r, pct in room_pct.items():
    print(f'  {r:20s} {pct:5.1f}%')

print('\n=== Host Verification ===')
print((df['host_identity_verified'].value_counts(normalize=True) * 100).round(1))

---
## 2 · Price Distribution Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Histogram
axes[0].hist(df['price'], bins=50, color='#6366f1', edgecolor='#0f1117', alpha=0.85)
axes[0].axvline(df['price'].median(), color='#f59e0b', linestyle='--', linewidth=2, label=f'Median: ${df["price"].median():.0f}')
axes[0].axvline(df['price'].mean(), color='#f43f5e', linestyle='--', linewidth=2, label=f'Mean: ${df["price"].mean():.0f}')
axes[0].set_title('Price Distribution')
axes[0].set_xlabel('Price ($)')
axes[0].legend(fontsize=9)

# Box plot by borough
borough_order = df.groupby('neighbourhood_group')['price'].median().sort_values(ascending=False).index
colors = [BOROUGH_COLORS[b] for b in borough_order]
bp = axes[1].boxplot([df[df.neighbourhood_group==b]['price'] for b in borough_order],
                     labels=borough_order, patch_artist=True, showfliers=False)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color + '80')
    patch.set_edgecolor(color)
axes[1].set_title('Price by Borough (Box Plot)')
axes[1].set_ylabel('Price ($)')
axes[1].tick_params(axis='x', rotation=25)

# Price segments
bins = [0, 200, 500, 800, 1200]
labels_seg = ['Budget\n($50-200)', 'Mid\n($200-500)', 'Premium\n($500-800)', 'Luxury\n($800-1200)']
df['price_segment'] = pd.cut(df['price'], bins=bins, labels=labels_seg)
seg_counts = df['price_segment'].value_counts().reindex(labels_seg)
seg_colors = ['#10b981', '#06b6d4', '#f59e0b', '#f43f5e']
axes[2].bar(seg_counts.index, seg_counts.values, color=seg_colors, edgecolor='#0f1117', width=0.7)
for i, v in enumerate(seg_counts.values):
    axes[2].text(i, v + 500, f'{v:,}', ha='center', fontsize=9, color='white')
axes[2].set_title('Listings by Price Segment')
axes[2].set_ylabel('Count')

plt.tight_layout()
plt.savefig('eda_price_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n>> Insight: Prices are roughly uniformly distributed between $50-$1200.')
print(f'   Median (${df["price"].median():.0f}) ≈ Mean (${df["price"].mean():.0f}), indicating low skew.')

---
## 3 · Borough-Level Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Avg price by borough
borough_stats = df.groupby('neighbourhood_group').agg(
    avg_price=('price', 'mean'),
    count=('id', 'count'),
    avg_rating=('review_rate_number', 'mean')
).sort_values('avg_price', ascending=True)

colors = [BOROUGH_COLORS[b] for b in borough_stats.index]

axes[0].barh(borough_stats.index, borough_stats['avg_price'], color=colors, edgecolor='#0f1117', height=0.6)
for i, (idx, row) in enumerate(borough_stats.iterrows()):
    axes[0].text(row['avg_price'] + 5, i, f'${row["avg_price"]:.0f}', va='center', fontsize=10, color='white')
axes[0].set_title('Avg Nightly Price by Borough')
axes[0].set_xlabel('Price ($)')

# Listing count by borough
axes[1].barh(borough_stats.index, borough_stats['count'], color=colors, edgecolor='#0f1117', height=0.6)
for i, (idx, row) in enumerate(borough_stats.iterrows()):
    axes[1].text(row['count'] + 200, i, f'{row["count"]:,}', va='center', fontsize=10, color='white')
axes[1].set_title('Number of Listings by Borough')
axes[1].set_xlabel('Listings')

# Avg rating by borough
axes[2].barh(borough_stats.index, borough_stats['avg_rating'], color=colors, edgecolor='#0f1117', height=0.6)
for i, (idx, row) in enumerate(borough_stats.iterrows()):
    axes[2].text(row['avg_rating'] + 0.02, i, f'{row["avg_rating"]:.2f}', va='center', fontsize=10, color='white')
axes[2].set_title('Avg Review Rating by Borough')
axes[2].set_xlabel('Rating')
axes[2].set_xlim(left=3.0)

plt.tight_layout()
plt.savefig('eda_borough_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Written insights
top_borough = borough_stats['avg_price'].idxmax()
bot_borough = borough_stats['avg_price'].idxmin()
top_p = borough_stats.loc[top_borough, 'avg_price']
bot_p = borough_stats.loc[bot_borough, 'avg_price']
print(f'>> Insight: {top_borough} (${top_p:.0f}) is the most expensive borough.')
print(f'   {bot_borough} (${bot_p:.0f}) is the cheapest — a {((top_p/bot_p - 1)*100):.0f}% gap.')
print(f'   Manhattan + Brooklyn account for {(borough_stats.loc[["Manhattan","Brooklyn"],"count"].sum()/len(df)*100):.0f}% of all listings.')
top_rated = borough_stats['avg_rating'].idxmax()
print(f'   {top_rated} has the highest avg rating ({borough_stats.loc[top_rated, "avg_rating"]:.2f}) despite having the fewest listings.')

---
## 4 · Room Type Breakdown

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
room_counts = df['room_type'].value_counts()
room_colors = ['#6366f1', '#06b6d4', '#f59e0b', '#f43f5e']
wedges, texts, autotexts = axes[0].pie(room_counts, labels=room_counts.index,
    autopct='%1.1f%%', colors=room_colors, startangle=90,
    wedgeprops={'edgecolor': '#0f1117', 'linewidth': 2})
for t in autotexts: t.set_fontsize(10)
axes[0].set_title('Listing Share by Room Type')

# Avg price by room type
room_price = df.groupby('room_type')['price'].mean().sort_values(ascending=True)
room_c = [room_colors[room_counts.index.tolist().index(r)] for r in room_price.index]
axes[1].barh(room_price.index, room_price.values, color=room_c, edgecolor='#0f1117', height=0.55)
for i, (rt, p) in enumerate(room_price.items()):
    axes[1].text(p + 5, i, f'${p:.0f}', va='center', color='white', fontsize=10)
axes[1].set_title('Avg Price by Room Type')
axes[1].set_xlabel('Price ($)')

plt.tight_layout()
plt.savefig('eda_room_types.png', dpi=150, bbox_inches='tight')
plt.show()

entire_pct = room_counts['Entire home/apt'] / room_counts.sum() * 100
print(f'>> Insight: Entire home/apt dominates at {entire_pct:.1f}% of listings.')
print(f'   Hotel rooms are extremely rare ({room_counts["Hotel room"]} listings = {room_counts["Hotel room"]/len(df)*100:.1f}%).')
print(f'   Shared rooms are cheapest at ${room_price["Shared room"]:.0f}/night avg — budget travelers\' go-to.')

---
## 5 · Correlation Heatmap

In [ ]:
num_cols = ['price', 'service_fee', 'minimum_nights', 'number_of_reviews',
            'reviews_per_month', 'review_rate_number',
            'calculated_host_listings_count', 'availability_365']
short_labels = ['Price', 'Svc Fee', 'Min Nights', 'Num Reviews',
                'Reviews/Mo', 'Rating', 'Host Listings', 'Availability']

corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, linewidths=0.5, linecolor='#1a1a2e',
            xticklabels=short_labels, yticklabels=short_labels,
            cbar_kws={'shrink': 0.8}, ax=ax)
ax.set_title('Correlation Matrix — Numeric Features', pad=16)
plt.tight_layout()
plt.savefig('eda_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

print('>> Insight: Price and Service Fee have near-perfect correlation (0.999) — service fee is a fixed % of price.')
print('   No strong correlation between price and reviews/ratings — expensive listings are NOT rated higher.')
print('   This means price is driven by location & room type, not quality.')

---
## 6 · Availability & Review Patterns

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Availability distribution
axes[0].hist(df['availability_365'], bins=40, color='#10b981', edgecolor='#0f1117', alpha=0.85)
axes[0].axvline(df['availability_365'].median(), color='#f59e0b', linestyle='--', lw=2,
                label=f'Median: {df["availability_365"].median():.0f} days')
axes[0].set_title('Availability Distribution (days/year)')
axes[0].set_xlabel('Days Available')
axes[0].legend(fontsize=9)

# Reviews distribution (capped at 200 for readability)
axes[1].hist(df[df['number_of_reviews'] <= 200]['number_of_reviews'], bins=50,
             color='#06b6d4', edgecolor='#0f1117', alpha=0.85)
no_review_pct = (df['number_of_reviews'] == 0).sum() / len(df) * 100
axes[1].set_title(f'Review Count Distribution (≤200)')
axes[1].set_xlabel('Number of Reviews')
axes[1].annotate(f'{no_review_pct:.1f}% have\n0 reviews', xy=(0.6, 0.85),
                  xycoords='axes fraction', fontsize=10, color='#f59e0b',
                  bbox=dict(boxstyle='round,pad=0.3', facecolor='#1a1a2e', edgecolor='#f59e0b'))

# Avg price by cancellation policy
cancel_price = df.groupby('cancellation_policy')['price'].mean().sort_values()
cancel_colors = ['#10b981', '#f59e0b', '#f43f5e']
axes[2].bar(cancel_price.index, cancel_price.values, color=cancel_colors, edgecolor='#0f1117', width=0.5)
for i, (idx, v) in enumerate(cancel_price.items()):
    axes[2].text(i, v + 5, f'${v:.0f}', ha='center', color='white', fontsize=10)
axes[2].set_title('Avg Price by Cancellation Policy')
axes[2].set_ylabel('Price ($)')

plt.tight_layout()
plt.savefig('eda_availability_reviews.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'>> Insight: {no_review_pct:.1f}% of listings have zero reviews — likely inactive or brand-new hosts.')
print(f'   Median availability is {df["availability_365"].median():.0f} days/year — many hosts list part-time.')
print(f'   Strict cancellation listings are slightly pricier — premium hosts set tighter policies.')

---
## 7 · Top Neighbourhoods Deep Dive

In [ ]:
# Top 15 neighbourhoods by listing count
top_hoods = df['neighbourhood'].value_counts().head(15)
hood_prices = df[df['neighbourhood'].isin(top_hoods.index)].groupby('neighbourhood')['price'].mean()

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Count
colors_h = ['#6366f1' if top_hoods.index[i] in df[df.neighbourhood_group=='Manhattan']['neighbourhood'].values
             else '#06b6d4' for i in range(len(top_hoods))]
axes[0].barh(top_hoods.index[::-1], top_hoods.values[::-1],
             color=colors_h[::-1], edgecolor='#0f1117', height=0.65)
axes[0].set_title('Top 15 Neighbourhoods by Listings')
axes[0].set_xlabel('Number of Listings')
for i, v in enumerate(top_hoods.values[::-1]):
    axes[0].text(v + 30, i, f'{v:,}', va='center', fontsize=9, color='white')

# Price in those neighbourhoods
hood_p_sorted = hood_prices.reindex(top_hoods.index).sort_values(ascending=True)
colors_p = ['#6366f1' if h in df[df.neighbourhood_group=='Manhattan']['neighbourhood'].values
             else '#06b6d4' for h in hood_p_sorted.index]
axes[1].barh(hood_p_sorted.index, hood_p_sorted.values, color=colors_p, edgecolor='#0f1117', height=0.65)
axes[1].set_title('Avg Price in Top 15 Neighbourhoods')
axes[1].set_xlabel('Price ($)')
for i, v in enumerate(hood_p_sorted.values):
    axes[1].text(v + 3, i, f'${v:.0f}', va='center', fontsize=9, color='white')

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#6366f1', label='Manhattan'),
                   Patch(facecolor='#06b6d4', label='Brooklyn')]
axes[0].legend(handles=legend_elements, loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig('eda_top_neighbourhoods.png', dpi=150, bbox_inches='tight')
plt.show()

top1 = top_hoods.index[0]
print(f'>> Insight: {top1} has the most listings ({top_hoods.iloc[0]:,}).')
print(f'   The top 15 neighbourhoods are split between Manhattan (purple) and Brooklyn (cyan).')
print(f'   Bedford-Stuyvesant and Williamsburg lead Brooklyn; Harlem and Hell\'s Kitchen lead Manhattan.')

---
## 8 · Room Type × Borough Heatmap

In [ ]:
# Crosstab: avg price by room type × borough
cross = df.pivot_table(values='price', index='room_type', columns='neighbourhood_group', aggfunc='mean').round(0)

fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(cross, annot=True, fmt='.0f', cmap='YlOrRd', linewidths=1,
            linecolor='#0f1117', cbar_kws={'label': 'Avg Price ($)'}, ax=ax)
ax.set_title('Avg Price ($) — Room Type × Borough', pad=12)
ax.set_ylabel('')
ax.set_xlabel('')
plt.tight_layout()
plt.savefig('eda_room_borough_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print('>> Insight: Entire home/apt is the most expensive room type across ALL boroughs.')
print('   Hotel rooms appear only in Manhattan and Brooklyn with very limited supply.')

---
## Key Insights Summary

| # | Insight | Evidence |
|---|---|---|
| 1 | **Manhattan + Brooklyn dominate** the market | Together they hold ~83% of all listings |
| 2 | **Entire home/apt is king** | 52%+ of listings; commands highest avg price |
| 3 | **Price does not equal Quality** | Near-zero correlation between price and review ratings |
| 4 | **Service fee is a fixed markup** | 0.999 correlation with price (~20% of nightly rate) |
| 5 | **15%+ listings are never reviewed** | Indicates inactive hosts or very new listings |
| 6 | **Staten Island: hidden gem** | Fewest listings but highest avg review rating |
| 7 | **Strict cancellation = higher price** | Premium hosts charge more and set tighter policies |
| 8 | **Top neighbourhoods cluster** in Manhattan and Brooklyn | Bedford-Stuyvesant, Williamsburg, Harlem, Hells Kitchen |

These insights are visualized in the Tableau dashboard for interactive exploration.